# Ejemplos Adicionales de Árboles de Decisión

Este notebook demuestra aplicaciones adicionales de Árboles de Decisión para tareas de clasificación y regresión, utilizando diferentes conjuntos de datos que los de los ejemplos originales.

## Configuración

Primero, importemos las bibliotecas necesarias y configuremos el entorno.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.tree import DecisionTreeClassifier, DecisionTreeRegressor
from sklearn.datasets import load_wine, fetch_california_housing
from sklearn.model_selection import train_test_split, GridSearchCV, cross_val_score
from sklearn.metrics import accuracy_score, confusion_matrix, mean_squared_error, r2_score
from sklearn.preprocessing import StandardScaler

np.random.seed(42)

plt.style.use('seaborn-v0_8-whitegrid')
plt.rc('font', size=12)
plt.rc('axes', labelsize=12, titlesize=14)
plt.rc('legend', fontsize=12)

In [ ]:
plt.rc('font', size=14)
plt.rc('axes', labelsize=14, titlesize=14)
plt.rc('legend', fontsize=14)
plt.rc('xtick', labelsize=10)
plt.rc('ytick', labelsize=10)

from pathlib import Path

RUTA_IMAGENES = Path() / "imagenes" / "arboles_decision"
RUTA_IMAGENES.mkdir(parents=True, exist_ok=True)

def guardar_fig(id_fig, tight_layout=True, extension_fig="png", resolucion=300):
    ruta = RUTA_IMAGENES / f"{id_fig}.{extension_fig}"
    if tight_layout:
        plt.tight_layout()
    plt.savefig(ruta, format=extension_fig, dpi=resolucion)

# Parte 1: Clasificación con Árboles de Decisión en el Conjunto de Datos Wine

Utilizaremos el conjunto de datos Wine de scikit-learn, que es diferente del conjunto de datos Iris utilizado en los ejemplos originales. El conjunto de datos Wine contiene los resultados de un análisis químico de vinos cultivados en una zona específica de Italia, con 13 características y 3 clases que representan diferentes cultivares.

## 1.1 Carga y Exploración del Conjunto de Datos Wine

In [ ]:
wine = load_wine()
X_wine = wine.data
y_wine = wine.target

print(f"Dataset shape: {X_wine.shape}")
print(f"Number of classes: {len(np.unique(y_wine))}")
print(f"Class distribution: {[np.sum(y_wine == i) for i in range(3)]}")

Dataset shape: (178, 13)
Number of classes: 3
Class distribution: [np.int64(59), np.int64(71), np.int64(48)]


In [ ]:
print("Feature names:")
print(wine.feature_names)
print("\nTarget names:")
print([f'class_{i}' for i in range(len(wine.target_names))])

Feature names:
['alcohol', 'malic_acid', 'ash', 'alcalinity_of_ash', 'magnesium', 'total_phenols', 'flavanoids', 'nonflavanoid_phenols', 'proanthocyanins', 'color_intensity', 'hue', 'od280/od315_of_diluted_wines', 'proline']

Target names:
['class_0', 'class_1', 'class_2']


## 1.2 División de los Datos

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X_wine, y_wine, test_size=0.2, random_state=42, stratify=y_wine
)

print(f"Training set shape: {X_train.shape}")
print(f"Test set shape: {X_test.shape}")

Training set shape: (142, 13)
Test set shape: (36, 13)


## 1.3 Entrenamiento de un Clasificador Básico de Árbol de Decisión

In [ ]:
wine_tree = DecisionTreeClassifier(random_state=42)
wine_tree.fit(X_train, y_train)

y_train_pred = wine_tree.predict(X_train)
y_test_pred = wine_tree.predict(X_test)

print(f"Training accuracy: {accuracy_score(y_train, y_train_pred):.3f}")
print(f"Test accuracy: {accuracy_score(y_test, y_test_pred):.3f}")

Training accuracy: 1.000
Test accuracy: 0.944


El árbol de decisión predeterminado logra una precisión perfecta en el conjunto de entrenamiento pero una precisión menor en el conjunto de prueba, lo que sugiere sobreajuste. Ajustemos los hiperparámetros para mejorar la generalización.

## 1.4 Ajuste de Hiperparámetros usando GridSearchCV

In [ ]:
param_grid = {
    'max_depth': [None, 3, 4, 5, 6],
    'min_samples_split': [2, 5, 10],
    'min_samples_leaf': [1, 2, 4]
}

grid_search = GridSearchCV(
    DecisionTreeClassifier(random_state=42),
    param_grid,
    cv=5,
    scoring='accuracy',
    n_jobs=-1
)

grid_search.fit(X_train, y_train)

print(f"Best parameters: {grid_search.best_params_}")
print(f"Best cross-validation score: {grid_search.best_score_:.3f}")

Best parameters: {'max_depth': 3, 'min_samples_leaf': 2, 'min_samples_split': 2}
Best cross-validation score: 0.930


## 1.5 Evaluación del Modelo Ajustado

In [ ]:
best_wine_tree = grid_search.best_estimator_

y_train_pred = best_wine_tree.predict(X_train)
y_test_pred = best_wine_tree.predict(X_test)

train_accuracy = accuracy_score(y_train, y_train_pred)
test_accuracy = accuracy_score(y_test, y_test_pred)
conf_matrix = confusion_matrix(y_test, y_test_pred)

print(f"Training accuracy: {train_accuracy:.3f}")
print(f"Test accuracy: {test_accuracy:.3f}")
print("\nConfusion matrix:")
print(conf_matrix)

Training accuracy: 0.986
Test accuracy: 1.000

Confusion matrix:
[[12  0  0]
 [ 0 14  0]
 [ 0  0 10]]


In [ ]:
from sklearn.tree import export_graphviz
from graphviz import Source

In [ ]:
export_graphviz(
    best_wine_tree,
    out_file=str(RUTA_IMAGENES / "arbol_vino.dot"),
    feature_names=wine.feature_names,
    class_names=[f'clase_{i}' for i in range(3)],
    rounded=True,
    filled=True
)

Source.from_file(RUTA_IMAGENES / "arbol_vino.dot")

!dot -Tpng {RUTA_IMAGENES / "arbol_vino.dot"} -o {RUTA_IMAGENES / "arbol_vino.png"}

## 1.6 Análisis de Importancia de Características

In [ ]:
importances = best_wine_tree.feature_importances_
indices = np.argsort(importances)[::-1]

print("Feature importances:")
for idx in indices:
    if importances[idx] > 0:
        print(f"{wine.feature_names[idx]}: {importances[idx]:.3f}")
    else:
        print(f"{wine.feature_names[idx]}: {importances[idx]:.3f}")

Feature importances:
flavanoids: 0.431
color_intensity: 0.428
proline: 0.114
total_phenols: 0.026
od280/od315_of_diluted_wines: 0.000
proanthocyanins: 0.000
hue: 0.000
nonflavanoid_phenols: 0.000
magnesium: 0.000
alcalinity_of_ash: 0.000
ash: 0.000
malic_acid: 0.000
alcohol: 0.000


In [ ]:
from matplotlib.colors import ListedColormap
cmap_personalizado = ListedColormap(['#fafab0', '#9898ff', '#a0faa0'])

In [ ]:
indices_ordenados = np.argsort(best_wine_tree.feature_importances_)[::-1]
caracteristicas = [wine.feature_names[i] for i in indices_ordenados[:2]]
X_vis = X_wine[:, indices_ordenados[:2]]

def visualizar_frontera_decision(clf, X, y, ejes, nombres_caract, cmapa):
    x1s = np.linspace(ejes[0], ejes[1], 100)
    x2s = np.linspace(ejes[2], ejes[3], 100)
    x1, x2 = np.meshgrid(x1s, x2s)
    X_nuevo = np.c_[x1.ravel(), x2.ravel()]
    y_pred = clf.predict(X_nuevo).reshape(x1.shape)

    plt.contourf(x1, x2, y_pred, alpha=0.3, cmap=cmapa)
    plt.contour(x1, x2, y_pred, cmap="Greys", alpha=0.8)

    marcadores = ("o", "^", "s")
    for idx in range(3):
        plt.plot(X[:, 0][y == idx], X[:, 1][y == idx],
                 marker=marcadores[idx], linestyle="none",
                 label=f"Clase {idx}")

    plt.xlabel(nombres_caract[0])
    plt.ylabel(nombres_caract[1])
    plt.axis(ejes)
    plt.legend()

fig, ejes = plt.subplots(ncols=2, figsize=(12, 5))

plt.sca(ejes[0])
arbol_vis_sin_restriccion = DecisionTreeClassifier(random_state=42)
arbol_vis_sin_restriccion.fit(X_vis, y_wine)
min_x1, max_x1 = X_vis[:, 0].min()-1, X_vis[:, 0].max()+1
min_x2, max_x2 = X_vis[:, 1].min()-1, X_vis[:, 1].max()+1
visualizar_frontera_decision(arbol_vis_sin_restriccion, X_vis, y_wine,
                           [min_x1, max_x1, min_x2, max_x2],
                           caracteristicas, cmap_personalizado)
plt.title("Árbol sin restricciones")

plt.sca(ejes[1])
arbol_vis_con_restriccion = DecisionTreeClassifier(max_depth=best_wine_tree.max_depth, random_state=42)
arbol_vis_con_restriccion.fit(X_vis, y_wine)
visualizar_frontera_decision(arbol_vis_con_restriccion, X_vis, y_wine,
                           [min_x1, max_x1, min_x2, max_x2],
                           caracteristicas, cmap_personalizado)
plt.title(f"Árbol podado (max_depth={best_wine_tree.max_depth})")

guardar_fig("fronteras_decision_vino")
plt.show()

In [ ]:
plt.figure(figsize=(10, 6))
importancias_ordenadas = sorted(zip(wine.feature_names, best_wine_tree.feature_importances_),
                               key=lambda x: x[1], reverse=True)
caract, importancia = zip(*importancias_ordenadas)

plt.bar(range(len(caract)), importancia, color='#9898ff')
plt.xticks(range(len(caract)), caract, rotation=90)
plt.title('Importancia de Características - Clasificación de Vinos')
plt.xlabel('Características')
plt.ylabel('Importancia')
plt.axhline(y=0.01, color='r', linestyle='--',
            label='Umbral de importancia (1%)')
plt.legend()
guardar_fig("importancia_caracteristicas_vino")
plt.show()

## 1.7 Resumen de Clasificación

Nuestro análisis del conjunto de datos Wine con Árboles de Decisión reveló resultados importantes para la toma de decisiones en la clasificación de vinos:

1. **Rendimiento del modelo predeterminado**: El árbol de decisión no ajustado logró una precisión perfecta en el entrenamiento (1.000) pero una precisión de prueba menor (0.944), lo que indica cierto nivel de sobreajuste. Esto sugiere que un sistema de toma de decisiones automático sin ajustes podría cometer errores innecesarios al clasificar nuevas muestras.

2. **Ajuste de hiperparámetros**: La búsqueda de cuadrícula encontró parámetros óptimos (max_depth=3, min_samples_leaf=2, min_samples_split=2) con una puntuación de validación cruzada de 0.930. Este proceso de optimización es fundamental para sistemas de soporte a la decisión, ya que maximiza la confiabilidad de las clasificaciones futuras.

3. **Importancia de características para la decisión**: Las características más importantes identificadas:
   - flavanoids (43.1%)
   - color_intensity (42.8%)
   - proline (11.4%)
   - total_phenols (2.6%)
   
   Estas características clave pueden utilizarse para desarrollar pruebas rápidas y económicas que faciliten decisiones de clasificación de vinos, priorizando las mediciones que aportan mayor valor informativo.

4. **Implicaciones para la toma de decisiones**: El árbol podado (profundidad=3) no solo es más interpretable, sino que también logró una precisión perfecta en el conjunto de prueba (1.000). Esto tiene dos importantes implicaciones para la toma de decisiones:
   - **Eficiencia operativa**: Las empresas vinícolas pueden implementar algoritmos de clasificación simplificados que requieran menos recursos computacionales.
   - **Transparencia decisional**: Un árbol con solo 3 niveles de profundidad facilita la explicación de decisiones a partes interesadas no técnicas (como enólogos, distribuidores o reguladores).

5. **Protocolo de decisión recomendado**: Basado en nuestros resultados, el proceso óptimo para clasificar vinos sería:
   1. Medir primero los flavanoides (característica más discriminativa)
   2. Seguido por la intensidad del color
   3. Finalmente, medir la prolina en casos ambiguos

   Este enfoque secuencial reduce el tiempo y costo del proceso de clasificación, maximizando la eficiencia en la toma de decisiones.

# Parte 2: Regresión con Árbol de Decisión en el Conjunto de Datos California Housing

Para el ejemplo de regresión, utilizaremos el conjunto de datos California Housing de scikit-learn, que es diferente del conjunto de datos cuadrático sintético utilizado en los ejemplos originales. Este conjunto de datos contiene información sobre precios de viviendas en diferentes distritos de California, con características como ingreso medio, edad de la vivienda y ubicación.

## 2.1 Carga y Exploración del Conjunto de Datos California Housing

In [ ]:
housing = fetch_california_housing()
X_housing = housing.data
y_housing = housing.target

print(f"Dataset shape: {X_housing.shape}")
print(f"Target statistics:")
print(f"- Min: {y_housing.min():.2f}")
print(f"- Max: {y_housing.max():.2f}")
print(f"- Mean: {y_housing.mean():.2f}")
print(f"- Median: {np.median(y_housing):.2f}")

Dataset shape: (20640, 8)
Target statistics:
- Min: 0.15
- Max: 5.00
- Mean: 2.07
- Median: 1.80


In [ ]:
print("Feature names:")
print(housing.feature_names)
print("\nTarget name: MedHouseVal (median house value in $100,000s)")

Feature names:
['MedInc', 'HouseAge', 'AveRooms', 'AveBedrms', 'Population', 'AveOccup', 'Latitude', 'Longitude']

Target name: MedHouseVal (median house value in $100,000s)


## 2.2 División de los Datos y Escalado de Características

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X_housing, y_housing, test_size=0.2, random_state=42
)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print(f"Training set shape: {X_train.shape}")
print(f"Test set shape: {X_test.shape}")

Training set shape: (16512, 8)
Test set shape: (4128, 8)


## 2.3 Entrenamiento de un Regresor Básico de Árbol de Decisión

In [ ]:
housing_tree = DecisionTreeRegressor(random_state=42)
housing_tree.fit(X_train, y_train)

y_train_pred = housing_tree.predict(X_train)
y_test_pred = housing_tree.predict(X_test)

train_r2 = r2_score(y_train, y_train_pred)
test_r2 = r2_score(y_test, y_test_pred)
train_rmse = np.sqrt(mean_squared_error(y_train, y_train_pred))
test_rmse = np.sqrt(mean_squared_error(y_test, y_test_pred))

print(f"Training R²: {train_r2:.3f}")
print(f"Test R²: {test_r2:.3f}")
print(f"Training RMSE: {train_rmse:.3f}")
print(f"Test RMSE: {test_rmse:.3f}")

Training R²: 1.000
Test R²: 0.622
Training RMSE: 0.000
Test RMSE: 0.704


El regresor de árbol de decisión predeterminado muestra un sobreajuste severo, con un R² perfecto en el conjunto de entrenamiento pero un rendimiento mucho menor en el conjunto de prueba. Esta es una señal clásica de que necesitamos aplicar regularización.

## 2.4 Ajuste de Hiperparámetros para Regresión

In [ ]:
param_grid = {
    'max_depth': [6, 8, 10, 12],
    'min_samples_split': [2, 5, 10],
    'min_samples_leaf': [5, 10, 20]
}

grid_search = GridSearchCV(
    DecisionTreeRegressor(random_state=42),
    param_grid,
    cv=5,
    scoring='r2',
    n_jobs=-1
)

grid_search.fit(X_train, y_train)

print(f"Best parameters: {grid_search.best_params_}")
print(f"Best cross-validation score (R²): {grid_search.best_score_:.3f}")

Best parameters: {'max_depth': 12, 'min_samples_leaf': 20, 'min_samples_split': 2}
Best cross-validation score (R²): 0.725


## 2.5 Evaluación del Modelo de Regresión Ajustado

In [ ]:
best_housing_tree = grid_search.best_estimator_

y_train_pred = best_housing_tree.predict(X_train)
y_test_pred = best_housing_tree.predict(X_test)

train_r2 = r2_score(y_train, y_train_pred)
test_r2 = r2_score(y_test, y_test_pred)
train_rmse = np.sqrt(mean_squared_error(y_train, y_train_pred))
test_rmse = np.sqrt(mean_squared_error(y_test, y_test_pred))

print(f"Training R²: {train_r2:.3f}")
print(f"Test R²: {test_r2:.3f}")
print(f"Training RMSE: {train_rmse:.3f}")
print(f"Test RMSE: {test_rmse:.3f}")

Training R²: 0.804
Test R²: 0.724
Training RMSE: 0.512
Test RMSE: 0.601


In [ ]:
def visualizar_predicciones_regresion(tree_reg, X, y, ejes=None, profundidades=None):
    if ejes is None:
        x_min, x_max = X[:, 0].min() - 1, X[:, 0].max() + 1
        y_min, y_max = y.min() - 0.1, y.max() + 0.1
        ejes = [x_min, x_max, y_min, y_max]

    caracteristica = np.argmax(tree_reg.feature_importances_)

    x_vis = np.linspace(ejes[0], ejes[1], 500).reshape(-1, 1)
    X_temp = np.zeros((x_vis.shape[0], X.shape[1]))
    for i in range(X.shape[1]):
        if i != caracteristica:
            X_temp[:, i] = X[:, i].mean()
    X_temp[:, caracteristica] = x_vis[:, 0]

    y_pred = tree_reg.predict(X_temp)

    plt.axis(ejes)
    plt.xlabel(f"{housing.feature_names[caracteristica]}")
    plt.ylabel("Valor de vivienda (en $100,000)")

    indices_muestra = np.random.randint(0, X.shape[0], size=200)
    plt.plot(X[indices_muestra, caracteristica], y[indices_muestra], "b.", alpha=0.5)
    plt.plot(x_vis, y_pred, "r.-", linewidth=2, label=r"$\hat{y}$")

    if profundidades is not None:
        umbrales = tree_reg.tree_.threshold
        umbrales_ordenados = sorted([t for t in umbrales if t != -2])[:6]
        estilos = ["k-", "k--", "k:", "k-.", "k-", "k--"]
        for umbral, estilo in zip(umbrales_ordenados, estilos):
            plt.plot([umbral, umbral], [ejes[2], ejes[3]], estilo, linewidth=1)

    plt.legend()

fig, ejes = plt.subplots(ncols=2, figsize=(14, 6))

plt.sca(ejes[0])
arbol_housing_simple = DecisionTreeRegressor(max_depth=2, random_state=42)
arbol_housing_simple.fit(X_train, y_train)
visualizar_predicciones_regresion(arbol_housing_simple, X_train, y_train, profundidades=True)
plt.title("Regresión con profundidad = 2")


plt.sca(ejes[1])
visualizar_predicciones_regresion(best_housing_tree, X_train, y_train)
plt.title(f"Regresión con profundidad = {best_housing_tree.max_depth}")

guardar_fig("regresion_viviendas_california")
plt.show()

El modelo ajustado muestra una generalización mucho mejor con una mejora significativa en la puntuación R² de prueba (de 0,622 a 0,724) y un RMSE reducido (de 0,704 a 0,601).

## 2.6 Importancia de Características en Regresión

In [ ]:
importances = best_housing_tree.feature_importances_
indices = np.argsort(importances)[::-1]

print("Feature importances:")
for idx in indices:
    print(f"{housing.feature_names[idx]}: {importances[idx]:.3f}")

Feature importances:
MedInc: 0.630
AveOccup: 0.136
Latitude: 0.076
Longitude: 0.064
HouseAge: 0.044
AveRooms: 0.035
AveBedrms: 0.009
Population: 0.006


## 2.7 Análisis del Efecto de la Profundidad del Árbol

In [ ]:
depths = [2, 3, 4, 5, 6, 8, 10, 15, 20, None]
train_scores = []
test_scores = []
train_rmses = []
test_rmses = []

for depth in depths:
    tree_reg = DecisionTreeRegressor(max_depth=depth, random_state=42)
    tree_reg.fit(X_train, y_train)

    y_train_pred = tree_reg.predict(X_train)
    y_test_pred = tree_reg.predict(X_test)

    train_r2 = r2_score(y_train, y_train_pred)
    test_r2 = r2_score(y_test, y_test_pred)
    train_rmse = np.sqrt(mean_squared_error(y_train, y_train_pred))
    test_rmse = np.sqrt(mean_squared_error(y_test, y_test_pred))

    train_scores.append(train_r2)
    test_scores.append(test_r2)
    train_rmses.append(train_rmse)
    test_rmses.append(test_rmse)

    print(f"Depth: {depth}, Train R²: {train_r2:.3f}, Test R²: {test_r2:.3f}, Train RMSE: {train_rmse:.3f}, Test RMSE: {test_rmse:.3f}")

Depth: 2, Train R²: 0.452, Test R²: 0.424, Train RMSE: 0.856, Test RMSE: 0.868
Depth: 3, Train R²: 0.538, Test R²: 0.510, Train RMSE: 0.786, Test RMSE: 0.802
Depth: 4, Train R²: 0.589, Test R²: 0.554, Train RMSE: 0.742, Test RMSE: 0.764
Depth: 5, Train R²: 0.638, Test R²: 0.600, Train RMSE: 0.696, Test RMSE: 0.724
Depth: 6, Train R²: 0.678, Test R²: 0.621, Train RMSE: 0.656, Test RMSE: 0.705
Depth: 8, Train R²: 0.760, Test R²: 0.678, Train RMSE: 0.566, Test RMSE: 0.650
Depth: 10, Train R²: 0.835, Test R²: 0.683, Train RMSE: 0.470, Test RMSE: 0.645
Depth: 15, Train R²: 0.961, Test R²: 0.646, Train RMSE: 0.229, Test RMSE: 0.681
Depth: 20, Train R²: 0.996, Test R²: 0.627, Train RMSE: 0.076, Test RMSE: 0.699
Depth: None, Train R²: 1.000, Test R²: 0.622, Train RMSE: 0.000, Test RMSE: 0.704


In [ ]:
profundidades = [2, 3, 4, 5, 6, 8, 10, 15, 20, None]
nombres_prof = [str(d) if d is not None else "Sin límite" for d in profundidades]
valores_prof = [d if d is not None else 25 for d in profundidades]

puntuaciones_train = []
puntuaciones_test = []
rmse_train = []
rmse_test = []

for prof in profundidades:
    arbol_reg = DecisionTreeRegressor(max_depth=prof, random_state=42)
    arbol_reg.fit(X_train, y_train)

    y_train_pred = arbol_reg.predict(X_train)
    y_test_pred = arbol_reg.predict(X_test)

    train_r2 = r2_score(y_train, y_train_pred)
    test_r2 = r2_score(y_test, y_test_pred)
    train_error = np.sqrt(mean_squared_error(y_train, y_train_pred))
    test_error = np.sqrt(mean_squared_error(y_test, y_test_pred))

    puntuaciones_train.append(train_r2)
    puntuaciones_test.append(test_r2)
    rmse_train.append(train_error)
    rmse_test.append(test_error)

plt.figure(figsize=(10, 5))
plt.plot(valores_prof, puntuaciones_train, "bo-", linewidth=2, label="Entrenamiento")
plt.plot(valores_prof, puntuaciones_test, "ro-", linewidth=2, label="Prueba")
plt.xlabel("Profundidad máxima")
plt.ylabel("Puntuación R²")
plt.title("Rendimiento vs. Profundidad del Árbol")
plt.xticks(valores_prof, nombres_prof, rotation=45)
plt.grid(True)
plt.legend()
guardar_fig("rendimiento_vs_profundidad_r2")
plt.show()

plt.figure(figsize=(10, 5))
plt.plot(valores_prof, rmse_train, "bo-", linewidth=2, label="RMSE Entrenamiento")
plt.plot(valores_prof, rmse_test, "ro-", linewidth=2, label="RMSE Prueba")
plt.xlabel("Profundidad máxima")
plt.ylabel("RMSE")
plt.title("Error vs. Profundidad del Árbol")
plt.xticks(valores_prof, nombres_prof, rotation=45)
plt.grid(True)
plt.legend()
guardar_fig("rendimiento_vs_profundidad_rmse")
plt.show()

In [ ]:
plt.figure(figsize=(12, 8))

scatter = plt.scatter(X_test[:, 6], X_test[:, 7],
                    c=y_test,
                    cmap='viridis',
                    alpha=0.7,
                    s=20,
                    edgecolors='k')

colorbar = plt.colorbar(scatter, label='Precio real (en $100,000)')

scatter2 = plt.scatter(X_test[:, 6], X_test[:, 7],
                     c=best_housing_tree.predict(X_test),
                     cmap='plasma',
                     alpha=0.3,
                     s=20)

colorbar2 = plt.colorbar(scatter2, label='Precio predicho (en $100,000)')

plt.title('Distribución Geográfica: Predicciones vs Valores Reales')
plt.xlabel('Latitud')
plt.ylabel('Longitud')
plt.grid(True, alpha=0.3)
plt.tight_layout()
guardar_fig("distribucion_geografica_predicciones")
plt.show()

In [ ]:
modelo1 = DecisionTreeRegressor(random_state=42)
modelo2 = DecisionTreeRegressor(min_samples_leaf=10, random_state=42)
modelo3 = DecisionTreeRegressor(min_samples_split=20, random_state=42)

modelo1.fit(X_train, y_train)
modelo2.fit(X_train, y_train)
modelo3.fit(X_train, y_train)

x1 = np.linspace(X_train[:, 0].min(), X_train[:, 0].max(), 500).reshape(-1, 1)
X_vis = np.zeros((x1.shape[0], X_train.shape[1]))
for i in range(1, X_train.shape[1]):
    X_vis[:, i] = X_train[:, i].mean()
X_vis[:, 0] = x1[:, 0]

y_pred1 = modelo1.predict(X_vis)
y_pred2 = modelo2.predict(X_vis)
y_pred3 = modelo3.predict(X_vis)

fig, ejes = plt.subplots(ncols=3, figsize=(15, 5), sharey=True)

plt.sca(ejes[0])
indices_muestra = np.random.randint(0, X_train.shape[0], size=200)
plt.scatter(X_train[indices_muestra, 0], y_train[indices_muestra], alpha=0.2)
plt.plot(x1, y_pred1, "r.-", linewidth=2, label=r"$\hat{y}$")
plt.xlabel(housing.feature_names[0])
plt.ylabel("Precio ($100,000)")
plt.title("Sin restricciones")
plt.legend()

plt.sca(ejes[1])
plt.scatter(X_train[indices_muestra, 0], y_train[indices_muestra], alpha=0.2)
plt.plot(x1, y_pred2, "r.-", linewidth=2, label=r"$\hat{y}$")
plt.xlabel(housing.feature_names[0])
plt.title(f"min_samples_leaf={modelo2.min_samples_leaf}")

plt.sca(ejes[2])
plt.scatter(X_train[indices_muestra, 0], y_train[indices_muestra], alpha=0.2)
plt.plot(x1, y_pred3, "r.-", linewidth=2, label=r"$\hat{y}$")
plt.xlabel(housing.feature_names[0])
plt.title(f"min_samples_split={modelo3.min_samples_split}")

guardar_fig("comparacion_modelos_regularizacion")
plt.show()

## 2.8 Resumen de Regresión

Nuestro análisis del conjunto de datos California Housing con regresión de Árbol de Decisión reveló importantes hallazgos para la toma de decisiones en el mercado inmobiliario:

1. **Rendimiento del modelo predeterminado y riesgo decisional**: El árbol de decisión no ajustado mostró un sobreajuste extremo con un R² de entrenamiento perfecto (1.000) pero un R² de prueba deficiente (0.622) y un RMSE de prueba de 0.704. Este comportamiento representa un riesgo significativo en sistemas de apoyo a decisiones inmobiliarias, ya que podría conducir a estimaciones de precios excesivamente optimistas o pesimistas, con un margen de error promedio de $70,400.

2. **Optimización de decisiones mediante ajuste de hiperparámetros**: La búsqueda de cuadrícula encontró parámetros óptimos (max_depth=12, min_samples_leaf=20, min_samples_split=2) con una puntuación de validación cruzada de 0.725. Esta configuración representa el mejor equilibrio entre complejidad del modelo y capacidad predictiva, crucial para decisiones fiables sobre inversiones inmobiliarias.

3. **Factores determinantes para la toma de decisiones**: Las características más influyentes fueron:
   - Ingreso medio (MedInc): 63.0% - factor económico dominante
   - Ocupación promedio (AveOccup): 13.6% - factor demográfico
   - Ubicación (Latitud: 7.6%, Longitud: 6.4%) - factor geográfico
   
   Estos resultados permiten a inversores y desarrolladores inmobiliarios priorizar variables clave al tomar decisiones de inversión o desarrollo.

4. **Umbral crítico de complejidad para decisiones óptimas**: El análisis mostró un rendimiento máximo de prueba a una profundidad de 10 (R²=0.683, RMSE=0.645), evidenciando que existe un punto de equilibrio óptimo en la complejidad del modelo. Esto sugiere que los sistemas de apoyo a decisiones inmobiliarias deberían limitarse a considerar aproximadamente 10 niveles de división en sus árboles de decisión.

5. **Implicaciones para la precisión de decisiones**: Con el modelo ajustado, se logró reducir el error promedio de predicción a $60,100 (RMSE=0.601), una mejora del 14.6% respecto al modelo no ajustado. Este nivel de precisión permite a los tomadores de decisiones establecer márgenes de seguridad apropiados en valuaciones inmobiliarias.

6. **Marco de decisión recomendado**: Basado en nuestros hallazgos, recomendamos un proceso secuencial para la valoración inmobiliaria:
   1. Evaluar primero los indicadores socioeconómicos (ingreso medio del área)
   2. Analizar factores de ocupación y densidad poblacional
   3. Considerar la ubicación geográfica específica
   4. Incluir características de la propiedad como refinamiento final
   
   Este enfoque estructurado permite optimizar recursos en el proceso de valoración mientras mantiene una alta precisión predictiva.

# Informe Completo: Árboles de Decisión para Clasificación y Regresión

## Introducción

Este informe examina la aplicación de Árboles de Decisión como sistemas inteligentes para la toma de decisiones, aplicados a problemas de clasificación y regresión utilizando conjuntos de datos del mundo real. Los Árboles de Decisión representan una metodología ideal para implementar sistemas de apoyo a decisiones gracias a su estructura intuitiva y transparente que emula el proceso de razonamiento humano. Investigamos su rendimiento, ajuste de hiperparámetros y análisis de importancia de características en el conjunto de datos Wine (clasificación) y el conjunto de datos California Housing (regresión).

## Metodología para Sistemas de Decisión

Para ambas tareas, seguimos una metodología sistemática enfocada en desarrollar sistemas confiables de toma de decisiones:
1. Exploración y preprocesamiento de datos para garantizar la calidad de entrada al sistema decisional
2. Entrenamiento de modelos de referencia para establecer una línea base de rendimiento decisional
3. Optimización de parámetros mediante GridSearchCV para maximizar la confiabilidad de las decisiones
4. Evaluación rigurosa en datos independientes para verificar la robustez del sistema
5. Análisis de importancia de variables para identificar los factores críticos en el proceso de decisión

## Sistemas de Clasificación Inteligentes (Conjunto de Datos Wine)

El problema de clasificación del conjunto de datos Wine demostró cómo los árboles de decisión pueden convertirse en sistemas efectivos de clasificación automática:

- **Estructura decisional óptima**: Un árbol con profundidad limitada a 3 niveles logró capturar perfectamente la lógica de clasificación, alcanzando un 100% de precisión en datos de prueba. Esto ilustra que sistemas de decisión relativamente simples pueden resolver problemas complejos cuando se optimizan adecuadamente.

- **Transparencia en la toma de decisiones**: A diferencia de otros algoritmos de "caja negra", el árbol de decisión optimizado ofrece total transparencia sobre sus criterios decisionales, priorizando mediciones de flavanoides (43.1%), intensidad del color (42.8%) y contenido de prolina (11.4%) como factores determinantes.

- **Protocolo decisional eficiente**: El análisis sugiere un protocolo de decisión secuencial que minimiza el número de pruebas necesarias para clasificar muestras, comenzando por las características más discriminativas y avanzando solo cuando sea necesario, lo que optimiza recursos en el proceso clasificatorio.

## Sistemas de Predicción para Decisiones de Inversión (Conjunto de Datos California Housing)

La tarea de predicción de precios de viviendas reveló cómo ajustar sistemas predictivos para la toma de decisiones financieras:

- **Calibración del nivel de complejidad decisional**: El análisis mostró que un sistema demasiado simple (profundidad <8) o demasiado complejo (profundidad >15) produce decisiones subóptimas. La profundidad ideal se situó entre 10-12 niveles de decisión, ofreciendo el mejor equilibrio entre precisión y generalización.

- **Jerarquía de factores decisionales**: El ingreso medio del área (63.0%) emergió como el factor dominante para las decisiones de valoración, seguido por factores demográficos y geográficos. Este hallazgo permite estructurar sistemas de decisión que prioricen la información más relevante.

- **Métricas para la confianza decisional**: La reducción del RMSE de $70,400 a $60,100 representa una mejora del 14.6% en la precisión predictiva, permitiendo establecer intervalos de confianza más ajustados en decisiones de inversión inmobiliaria.

## Análisis Comparativo de Sistemas Decisionales

La comparación entre ambos sistemas de decisión reveló patrones comunes relevantes para el diseño de sistemas inteligentes:

1. **Balance entre complejidad y robustez**: Ambos casos demostraron que sistemas de decisión más simples pero bien calibrados superan a sistemas complejos sin optimizar, confirmando el principio de parsimonia en el diseño de sistemas inteligentes.

2. **Importancia de la selección de variables**: En ambos dominios, un pequeño subconjunto de variables controló la mayoría de las decisiones, ilustrando el principio de Pareto en sistemas decisionales donde aproximadamente el 20% de las variables influyen en el 80% de los resultados.

3. **Transparencia como ventaja competitiva**: La interpretabilidad inherente de los árboles de decisión representa una ventaja crucial para sistemas de apoyo a decisiones en entornos regulados o de alto riesgo, donde es necesario explicar y justificar cada decisión tomada.

## Conclusiones y Recomendaciones para Sistemas de Toma de Decisiones

Basándonos en nuestro análisis, ofrecemos las siguientes recomendaciones para el desarrollo de sistemas inteligentes de toma de decisiones:

1. **Calibración sistemática**: El ajuste metódico de hiperparámetros no es un lujo sino una necesidad para garantizar decisiones confiables. Recomendamos implementar protocolos de validación cruzada para todos los sistemas decisionales críticos.

2. **Arquitectura jerárquica de decisiones**: Estructurar sistemas que evalúen primero las variables de mayor impacto, ramificándose progresivamente hacia factores secundarios solo cuando sea necesario, optimizando así tanto la velocidad como la precisión decisional.

3. **Evaluación continua**: Implementar métricas de rendimiento en tiempo real que monitoreen la divergencia entre predicciones y resultados reales, permitiendo recalibrar el sistema cuando se detecten desviaciones significativas.

4. **Hibridación estratégica**: Para decisiones de alta complejidad, considerar sistemas híbridos que combinen la interpretabilidad de los árboles de decisión con la potencia predictiva de otros algoritmos, asignando a cada uno el rol más apropiado en la cadena decisional.

Los árboles de decisión, a pesar de su aparente simplicidad, constituyen una herramienta poderosa y versátil para implementar sistemas inteligentes de apoyo a la toma de decisiones en diversos dominios, ofreciendo un equilibrio único entre capacidad predictiva, interpretabilidad y adaptabilidad a distintos contextos decisionales.